In [6]:
""" Get QoS """
import common_utils
import os
import pandas as pd
import difflib

# Example usage
root_folder = '../../../data_warehouse/minimized_warehouse_5b'
filename = 'worker1.feather'
subfolders = common_utils.find_subfolders_with_file(root_folder, filename)
print(subfolders)
prom_data_paths = {os.path.basename(x): x for x in subfolders}
yolo_data_paths = {key: os.path.join(val, "worker_qos.feather") for key, val in prom_data_paths.items()}


['../../../data_warehouse/minimized_warehouse_5b/1738216968_(9.10000)', '../../../data_warehouse/minimized_warehouse_5b/1738269051_(3.1000)', '../../../data_warehouse/minimized_warehouse_5b/1738209702_(9.1000)', '../../../data_warehouse/minimized_warehouse_5b/1738240668_(6.1000)', '../../../data_warehouse/minimized_warehouse_5b/1738256261_(5.10000)', '../../../data_warehouse/minimized_warehouse_5b/1738202557_(10.5000)', '../../../data_warehouse/minimized_warehouse_5b/1738198877_(10.1000)', '../../../data_warehouse/minimized_warehouse_5b/1738213376_(9.5000)', '../../../data_warehouse/minimized_warehouse_5b/1738286350_(1.1000)', '../../../data_warehouse/minimized_warehouse_5b/1738244204_(6.5000)', '../../../data_warehouse/minimized_warehouse_5b/1738259762_(4.1000)', '../../../data_warehouse/minimized_warehouse_5b/1738206191_(10.10000)', '../../../data_warehouse/minimized_warehouse_5b/1738250513_(5.1000)', '../../../data_warehouse/minimized_warehouse_5b/1738253453_(5.5000)', '../../../dat

In [7]:
"""
Get item count for each model and resolution
"""
num_items_dict = {}
for key in prom_data_paths.keys():
    try:
        yolo_df = common_utils.read_feather_cached(yolo_data_paths[key])
    except:
        print(f"Failed to read {key}")
        continue
    total_items = yolo_df["id"].max()
    model_info = common_utils.path_to_workers_and_pcl_size(key)
    if model_info.resolution not in num_items_dict:
        num_items_dict[model_info.resolution] = {}
    num_items_dict[model_info.resolution][model_info.num_vehicles] = total_items

In [8]:
print(num_items_dict.keys())
print(num_items_dict[10000].keys())
# %%
import plotly.express as px
import plotly.graph_objs as go
from plotly.subplots import make_subplots

# Prepare data for plotting
plot_data = []
for resolution, models in num_items_dict.items():
    for num_vehicles, num_items in models.items():
        plot_data.append({
            'Resolution': resolution,
            'Number of Vehicles': num_vehicles,
            'Number of Items': num_items
        })

# Convert to DataFrame
import pandas as pd

df = pd.DataFrame(plot_data)

# Create subplot for each resolution
fig = make_subplots(
    rows=len(num_items_dict.keys()),
    cols=1,
    subplot_titles=[f'Resolution: {resolution}' for resolution in sorted(num_items_dict.keys())]
)

# Color palette
colors = px.colors.qualitative.Plotly

# Add bar plots for each resolution
for i, resolution in enumerate(sorted(num_items_dict.keys()), 1):
    # Filter data for this resolution
    res_data = df[df['Resolution'] == resolution]

    # Create bar plot for this resolution
    bar = go.Bar(
        x=res_data['Number of Vehicles'],
        y=res_data['Number of Items'],
        name=f'Resolution {resolution}',
        marker_color=colors[i % len(colors)]
    )

    # Add to subplot
    fig.add_trace(bar, row=i, col=1)

# Update layout
fig.update_layout(
    title_text='Number of Items Processed by Model Configuration',
    height=300 * len(num_items_dict.keys()),
    width=800,
    showlegend=False
)

# Update axes
fig.update_xaxes(title_text='Number of Vehicles', row=len(num_items_dict.keys()), col=1)
fig.update_yaxes(title_text='Number of Items', row=1, col=1)

# Show the plot
fig.show()


dict_keys([10000, 1000, 5000])
dict_keys([9, 5, 10, 8, 3, 30, 6, 1, 2, 4, 20, 7])


In [9]:
# Helper for computing total joules from dataframe
cols = set()
num_cols = 2
def get_total_joules(dataframe):
    cleaned_df = dataframe
    
    """ Sort by timestamp to make sure it makes sense to compute difference between first and last values """
    cleaned_df.sort_values(by="timestamp", inplace=True)
    
    """ Get all relevant columns for power calculation """
    target_word = 'kepler node package joules total dynamic'
    closest_matches = difflib.get_close_matches(target_word, cleaned_df.columns, n=num_cols, cutoff=0.05)
    
    """ Compute joules per match """
    joules_per_match = []
    for match in closest_matches:
        joules = cleaned_df[match].max() - cleaned_df[match].min()
        joules_per_match.append(joules)
    cols.update(closest_matches)

    
    """ Compute total joules """
    total_joules = sum(joules_per_match)
    return total_joules

# Get total joules for each model
total_joules_per_model = {}
for key in prom_data_paths.keys():
    paths = []
    model_info = common_utils.path_to_workers_and_pcl_size(key)

    """ Get all workers """
    for work_num in range(1, 6):
        temp_path = os.path.join(prom_data_paths[key], f"worker{work_num}.feather")
        paths.append(temp_path)

    """ Get joules per image for each worker """
    joules_per_worker = [get_total_joules(common_utils.get_cleaned_df(x)) for x in paths]
    joules_total = sum(joules_per_worker)
    num_images = num_items_dict[model_info.resolution][model_info.num_vehicles]
    joules_per_image = joules_total / num_images

    """ Add result to dict for current model and resolution """

    if model_info.resolution not in total_joules_per_model:
        total_joules_per_model[model_info.resolution] = {}
    total_joules_per_model[model_info.resolution][model_info.num_vehicles] = joules_per_image

# Group the data by resolution
max_joules = {}
for resolution in sorted(total_joules_per_model.keys()):
    joules = pd.DataFrame.from_dict(total_joules_per_model[resolution], orient='index', columns=['Joules'])
    joules.columns = [f'{resolution}']
    max_joules[resolution] = joules

print(f"Used these {num_cols} cols: {cols}")


Used these 2 cols: {'kepler_node_package_joules_total_mode_idle', 'kepler_node_package_joules_total_mode_dynamic'}


In [10]:
from matplotlib import pyplot as plt
# Grouped bars
import plotly.express as px
import numpy as np

# Define width based on resolution
# resolution_to_width = {160: 0.2, 320: 0.4, 640: 0.6, 1280: 0.8}
max_joules_df = pd.concat(max_joules.values(), axis=1)
max_joules_df_sorted = max_joules_df.sort_index()  # Sort by index first

# Create separator rows with NaN values
separator_row1 = pd.DataFrame(index=["..."], columns=max_joules_df_sorted.columns, data=np.nan)
separator_row2 = pd.DataFrame(index=["...."], columns=max_joules_df_sorted.columns,
                              data=np.nan)  # Using .... to make it unique

# Split the dataframe into three parts and insert the separators
mask1 = max_joules_df_sorted.index <= 10
mask2 = (max_joules_df_sorted.index > 10) & (max_joules_df_sorted.index <= 20)
mask3 = max_joules_df_sorted.index > 20

df_part1 = max_joules_df_sorted[mask1]
df_part2 = max_joules_df_sorted[mask2]
df_part3 = max_joules_df_sorted[mask3]

# Combine all parts with the separators
max_joules_df_sorted = pd.concat([df_part1, separator_row1, df_part2, separator_row2, df_part3])

# Convert remaining numeric indices to strings
max_joules_df_sorted.index = max_joules_df_sorted.index.astype(str)

fig = px.bar(max_joules_df_sorted, barmode='group', title='Joules per PCL',
             labels={'value': 'Max Power (Watts)', 'index': 'Model'})
fig.update_layout(xaxis_title='Num_workers', yaxis_title='Joules', legend_title_text='Resolution',
                  xaxis={'categoryorder': 'array', 'categoryarray': max_joules_df_sorted.index},
                  title='5b (linear run)')
fig.show()

fig = px.bar(max_joules_df_sorted, barmode='group', title='Joules per PCL (Log Scale)',
             labels={'value': 'Max Power (Watts)', 'index': 'Model'})
fig.update_layout(xaxis_title='Num_workers', yaxis_title='Joules',
                  yaxis_type='log', legend_title_text='Resolution',
                  xaxis={'categoryorder': 'array', 'categoryarray': max_joules_df_sorted.index})
fig.show()
